# 08-2 Choropleth 地圖概念：GeoJSON + Plotly

上一個 notebook 分析了護理之家內部的空間分布。
本 notebook 延伸到 **地理空間** 層級，學習 choropleth（分級著色地圖）的製作原理。

這項技能在社區層級的疫調中非常實用——例如呈現各行政區的發生率。

流程：**讀取地區人口 → 計算發生率 → 讀取 GeoJSON → 畫 choropleth → ID 比對除錯**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 計算地區發生率 ---
import json
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Plotly: 確保在靜態建置（jupyter-book build）時也能輸出互動圖
pio.renderers.default = "notebook"

# 讀取人口資料
pop = pd.read_csv("data/synthetic/location_population.csv")
print("人口資料：")
print(pop)

# 讀取 line_list 病例（示範用，非 Legionella 資料集）
cases = (
    pd.read_csv("data/synthetic/line_list.csv")
    .groupby("location").size()
    .rename("cases").reset_index()
)

# 合併並計算發生率
rate = pop.merge(cases, on="location", how="left").fillna({"cases": 0})
rate["cases"] = rate["cases"].astype(int)
rate["incidence_per_100k"] = (rate["cases"] / rate["population"] * 100_000).round(1)

print("\n地區發生率：")
print(rate)

In [ ]:
# --- Step 2: 讀取 GeoJSON ---
with open("data/synthetic/admin_areas.geojson", "r", encoding="utf-8") as f:
    geojson = json.load(f)

# 檢查 feature 數量與 ID
features = geojson["features"]
print(f"GeoJSON 共 {len(features)} 個 feature")
for feat in features:
    loc = feat["properties"]["location"]
    geom_type = feat["geometry"]["type"]
    print(f"  {loc} ({geom_type})")

In [ ]:
# --- Step 3: 繪製 Choropleth ---
fig = px.choropleth(
    rate,
    geojson=geojson,
    locations="location",
    featureidkey="properties.location",
    color="incidence_per_100k",
    color_continuous_scale="Reds",
    title="Incidence per 100k by Location (Choropleth)",
    labels={"incidence_per_100k": "Incidence per 100k"},
)
fig.update_geos(fitbounds="locations", visible=False)
fig.show()

print("\u2192 顏色越深 = 發生率越高")
print("\u2192 這種圖適合在決策會議中快速呈現空間風險")

In [ ]:
# --- Step 4: ID 比對除錯（Debug Checklist）---
# 如果 choropleth 有空白區域，最常見的原因是 ID 不匹配

geo_ids = {feat["properties"]["location"].strip() for feat in geojson["features"]}
data_ids = set(rate["location"].astype(str).str.strip())

print("=== ID 比對 ===")
print(f"GeoJSON IDs: {sorted(geo_ids)}")
print(f"Data IDs:    {sorted(data_ids)}")
print(f"\nOnly in data (\u2192 地圖上不會顯示): {sorted(data_ids - geo_ids)}")
print(f"Only in GeoJSON (\u2192 地圖上沒顏色): {sorted(geo_ids - data_ids)}")
print(f"Missing rate values: {rate['incidence_per_100k'].isna().any()}")

if not (data_ids - geo_ids) and not (geo_ids - data_ids):
    print("\n\u2705 所有 ID 完全匹配！")

## 小結

| 概念 | 說明 |
|------|------|
| `GeoJSON` | 地理邊界資料格式，每個 feature 有 properties + geometry |
| `featureidkey` | GeoJSON 中用來匹配資料表的欄位路徑 |
| `px.choropleth()` | Plotly 的分級著色地圖函數 |
| ID 比對 | 製作 choropleth 前必做的除錯步驟 |

**Choropleth 適用場景**：
- 社區層級疫調（各行政區的發生率）
- 跨國疫情比較（如 COVID-19 全球地圖）
- 任何有地理邊界的資料

**本案場景**：松柏護理之家的空間分析主要在建築內部（floor × wing），
因此上一個 notebook 的 heatmap 和 spot map 更為實用。
但 choropleth 是空間流病的核心技能，在社區層級疫調中不可或缺。